# Customer Churn Intelligence — End-to-End Analysis
This notebook uses the reusable code in `src/` so the notebook and Streamlit app share the same training logic.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import plotly.express as px
from src.data import load_dataset
from src.modeling import train_and_evaluate, global_feature_importance, score_customers

## 1. Load and inspect data

In [ ]:
df = load_dataset()
print(df.shape)
display(df.head())
display(df['Churn'].value_counts(normalize=True).rename('share'))

## 2. Data quality checks

In [ ]:
print('Duplicates:', df.duplicated().sum())
print('Blank TotalCharges:', df['TotalCharges'].astype(str).str.strip().eq('').sum())
display(df.isna().sum().sort_values(ascending=False).head(10))

## 3. Business EDA

In [ ]:
eda = df.assign(ChurnFlag=df['Churn'].eq('Yes').astype(int))
contract = eda.groupby('Contract', as_index=False)['ChurnFlag'].mean()
px.bar(contract, x='Contract', y='ChurnFlag', title='Churn Rate by Contract')

## 4. Train, compare, and evaluate models
The pipeline uses an 80/20 stratified split, 5-fold CV, out-of-fold threshold tuning, and final held-out evaluation.

In [ ]:
bundle = train_and_evaluate(df)
display(bundle.comparison)
print('Selected:', bundle.best_model_name)
print('Threshold:', bundle.threshold)
display(pd.Series(bundle.metrics))

## 5. Global model drivers

In [ ]:
importance = global_feature_importance(bundle.model).head(20)
display(importance)
px.bar(importance.sort_values('importance'), x='importance', y='feature', orientation='h', title='Top Features')

## 6. Retention prioritization

In [ ]:
scored = score_customers(bundle.model, df.drop(columns=['Churn']))
display(scored[['customerID','ChurnProbability','RiskTier','AnnualCustomerValue','RevenueAtRisk']].head(20))

## 7. Interactive application
Run `streamlit run app.py` from the repository root for the executive dashboard, customer predictor, batch scorer, and model-insights views.